<a href="https://colab.research.google.com/github/ikny/toying-with-toy-models/blob/main/toy_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Replicating a part of Toy Models of Superposition
The code was taken from the original [notebook](https://colab.research.google.com/github/anthropics/toy-models-of-superposition/blob/main/toy_models.ipynb) and modified.

The paper examines how small models exploit feature sparsity to compress large feature space into a small activation space (representation). This compression gives rise to superposition and polysemantic neurons. It shows the models are not only able to recover the compressed input, but also to perform computation in superposition. For more detailed explanation refer to [the paper](https://transformer-circuits.pub/2022/toy_model/index.html) or [Neel Nanda's summary](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=EuO4CLwSIzX7AEZA1ZOsnwwF).

# Initialisation

## Toy Models of Superposition

This notebook includes the toy model training framework used to generate most of the results in the "Toy Models of Superposition" paper.

The main useful improvement over a basic PyTorch tiny autoencoder is the ability to batch train many models with varying sparsity at once, which is much more efficient than training them one at a time.

This notebook is designed to run in Google Colab's Python 3.7 environment.

In [2]:
!pip install einops

In [3]:
import torch
from torch import nn
from torch.nn import functional as F

from typing import Optional

from dataclasses import dataclass, replace
import numpy as np
import einops

from tqdm.notebook import trange

import time
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import matplotlib.pyplot as plt

In [4]:
@dataclass
class Config:
  n_features: int
  n_hidden: int

  # We optimize n_instances models in a single training loop
  # to let us sweep over sparsity or importance curves
  # efficiently.

  # We could potentially use torch.vmap instead.
  n_instances: int

class Model(nn.Module):
  def __init__(self,
               config,
               feature_probability: Optional[torch.Tensor] = None,
               importance: Optional[torch.Tensor] = None,
               device='cuda'):
    super().__init__()
    self.config = config
    self.W = nn.Parameter(torch.empty((config.n_instances, config.n_features, config.n_hidden), device=device))
    nn.init.xavier_normal_(self.W)
    self.b_final = nn.Parameter(torch.zeros((config.n_instances, config.n_features), device=device))

    if feature_probability is None:
      feature_probability = torch.ones(())
    self.feature_probability = feature_probability.to(device)
    if importance is None:
      importance = torch.ones(())
    self.importance = importance.to(device)

  def forward(self, features):
    # features: [..., instance, n_features]
    # W: [instance, n_features, n_hidden]
    hidden = torch.einsum("...if,ifh->...ih", features, self.W)
    out = torch.einsum("...ih,ifh->...if", hidden, self.W)
    out = out + self.b_final
    out = F.relu(out)
    return out

  def generate_batch(self, n_batch):
    feat = torch.rand((n_batch, self.config.n_instances, self.config.n_features), device=self.W.device)
    batch = torch.where(
        torch.rand((n_batch, self.config.n_instances, self.config.n_features), device=self.W.device) <= self.feature_probability,
        feat,
        torch.zeros((), device=self.W.device),
    )
    return batch

In [5]:
def linear_lr(step, steps):
  return (1 - (step / steps))

def constant_lr(*_):
  return 1.0

def cosine_decay_lr(step, steps):
  return np.cos(0.5 * np.pi * step / (steps - 1))

def optimize(model,
             render=False,
             n_batch=1024,
             steps=10_000,
             print_freq=100,
             lr=1e-3,
             lr_scale=constant_lr,
             hooks=[]):
  cfg = model.config

  opt = torch.optim.AdamW(list(model.parameters()), lr=lr)

  start = time.time()
  with trange(steps) as t:
    for step in t:
      step_lr = lr * lr_scale(step, steps)
      for group in opt.param_groups:
        group['lr'] = step_lr
      opt.zero_grad(set_to_none=True)
      batch = model.generate_batch(n_batch)
      out = model(batch)
      error = (model.importance*(batch.abs() - out)**2)
      loss = einops.reduce(error, 'b i f -> i', 'mean').sum()
      loss.backward()
      opt.step()

      if hooks:
        hook_data = dict(model=model,
                         step=step,
                         opt=opt,
                         error=error,
                         loss=loss,
                         lr=step_lr)
        for h in hooks:
          h(hook_data)
      if step % print_freq == 0 or (step + 1 == steps):
        t.set_postfix(
            loss=loss.item() / cfg.n_instances,
            lr=step_lr,
        )

In [6]:
if torch.cuda.is_available():
  DEVICE = 'cuda'
else:
  DEVICE = 'cpu'

# Visualizing features across varying sparsity
I aim to qualitatively reproduce the following figure from the original paper (this beautiful graphic is worth reading both for its own sake and for the sake of comparison with my later result):
![](https://raw.githubusercontent.com/ikny/toying-with-toy-models/main/assets/figure.png)

I changed the parameters (number of features, their importance and sparsity, number of hidden neurons) to fit the setup above. I disected a part of the code (see below), and added visualisation of the biases.

In [7]:
config = Config(
    n_features = 20,
    n_hidden = 5,
    n_instances = 7,
)

model = Model(
    config=config,
    device=DEVICE,
    # Exponential feature importance curve from 1 to 0.7**19
    importance = (0.7 ** torch.arange(config.n_features))[None, :],
    # Sweep feature frequency across the instances as in the figure
    feature_probability = torch.tensor((1., 0.3, 0.1, 0.03, 0.01, 0.003, 0.001))[:, None]
)

In [8]:
optimize(model)

  0%|          | 0/10000 [00:00<?, ?it/s]

In the section [basic results](https://transformer-circuits.pub/2022/toy_model/index.html#demonstrating-basic-results), a metric to measure interference of other features with feature $x_i$ is given: $\sum_{i\neq j} (\hat W_i\cdot W_j)^2$. However, in the figures legend, another metric is used: $\sum_j (\hat x_i\cdot x_j)^2$. This unclarity led me to examine the code below in more detail. I found out the code from the original notebook uses the first metric and I kept it.

Explanation of some of the code below:
- `W` is a 3D tensor, *not a matrix* as in the paper, which can be confusing at first. This is because we store the matrices of all the  models in it. Shape of `W` is `(n_instances, n_features, n_hidden)` (in the paper, shape of W is `(n_hidden, n_features)`).
- `W_norm = W / (1e-5 + torch.linalg.norm(W, 2, dim=-1, keepdim=True))`: for each model, we take the norms of the row vectors, which correspond to column vectors in the paper due to `W` here being transposed. I assume the `1e-5` is for numerical stability.
- The way interference is calculated in the code is consistent with the paper.

In [10]:
def render_features(model, which=np.s_[:]):
  cfg = model.config
  W = model.W.detach()
  W_norm = W / (1e-5 + torch.linalg.norm(W, 2, dim=-1, keepdim=True))

  interference = torch.einsum('ifh,igh->ifg', W_norm, W)
  interference[:, torch.arange(cfg.n_features), torch.arange(cfg.n_features)] = 0

  polysemanticity = torch.linalg.norm(interference, dim=-1).cpu()
  net_interference = (interference**2 * model.feature_probability[:, None, :]).sum(-1).cpu()
  norms = torch.linalg.norm(W, 2, dim=-1).cpu()

  WtW = torch.einsum('sih,soh->sio', W, W).cpu()

  # width = weights[0].cpu()
  # x = torch.cumsum(width+0.1, 0) - width[0]
  x = torch.arange(cfg.n_features)
  width = 0.9

  which_instances = np.arange(cfg.n_instances)[which]
  fig = make_subplots(rows=len(which_instances),
                      cols=3,
                      shared_xaxes=True,
                      vertical_spacing=0.02,
                      horizontal_spacing=0.1)
  for (row, inst) in enumerate(which_instances):
    fig.add_trace(
        go.Bar(x=x,
              y=norms[inst],
              marker=dict(
                  color=polysemanticity[inst],
                  cmin=0,
                  cmax=1
              ),
              width=width,
        ),
        row=1+row, col=1
    )
    data = WtW[inst].numpy()
    fig.add_trace(
        go.Image(
            z=plt.cm.coolwarm((1 + data)/2, bytes=True),
            colormodel='rgba256',
            customdata=data,
            hovertemplate='''\
In: %{x}<br>
Out: %{y}<br>
Weight: %{customdata:0.2f}
'''
        ),
        row=1+row, col=2
    )
    # add visualization of the biases
    data = model.b_final[inst].detach().cpu().numpy()[:,None]
    fig.add_trace(
        go.Image(
            z=plt.cm.coolwarm((1 + data)/2, bytes=True),
            colormodel='rgba256',
            customdata=data,
        ),
        row=1+row, col=3
    )


  fig.add_vline(
    x=(x[cfg.n_hidden-1]+x[cfg.n_hidden])/2,
    line=dict(width=0.5),
    col=1,
  )

  # fig.update_traces(marker_size=1)
  fig.update_layout(showlegend=False,
                    width=600,
                    height=100*len(which_instances),
                    margin=dict(t=0, b=0))
  fig.update_xaxes(visible=False)
  fig.update_yaxes(visible=False)
  return fig

In [11]:
fig = render_features(model)
fig.update_layout()

We can see that the crucial characteristics are reproduced:
- When features are dense, the model just represents the five most important features orthogonally (in separate dimensions).
- As features become sparser, superposition appears in the less important represented features, and progresses to the more important.
- The model also gradually represents more and more features.
- From second model (row) on, we can see *antipodal pairs*  forming (pairs of features having opposite representation).
- Biases: expected value (positive) in the beginning, shifts to negative values.

Interestingly, the first five features seem to be represented slightly less as sparsity increases. **This is different from the paper**, and I don't know why it happens.

# Questions
- I would be very interested to see [computation in superposition](https://transformer-circuits.pub/2022/toy_model/index.html#computation) demonstrated on a tiny model (~ three features, 4-5 neurons). What type of circuit would it implement for computing the absolute value with superposition? Something like this [is done](https://transformer-circuits.pub/2022/toy_model/index.html#computation-asymmetric-motif) in the paper, I would like to experiment with even smaller examples.
- How would a mathematical argument for "ReLU hidden layer $\implies$ model uses privileged basis" look like?
- The paper mentions *decomposability* as a crucial property of the activation of neural network.
  > Decomposability: Neural network activations which are decomposable can be decomposed into features, the meaning of which is not dependent on the value of other features.

  Is this necessarily a yes-or-no property? Would it make sense to think about partial decomposability?
- How does [Johnson–Lindenstrauss lemma](https://en.wikipedia.org/wiki/Johnson%E2%80%93Lindenstrauss_lemma) imply "it's possible to have $exp(n)$  many "almost orthogonal" ( $<ϵ$ cosine similarity) vectors in high-dimensional spaces"?